In [ ]:
import numpy as np
import pandas as pd
from pandasql import sqldf
import seaborn as sns
import matplotlib.pyplot as plt

base_path = "/kaggle/input/competitions/datathon-2026-round-1/"

products = pd.read_csv(
    base_path + "products.csv",
    dtype={
        "product_id": "Int64",
        "product_name": "string",
        "category": "string",
        "segment": "string",
        "size": "string",
        "color": "string",
        "price": "float64",
        "cogs": "float64",
    }
)

customers = pd.read_csv(
    base_path + "customers.csv",
    dtype={
        "customer_id": "Int64",
        "zip": "Int64",
        "city": "string",
        "gender": "string",
        "age_group": "string",
        "acquisition_channel": "string",
    },
    parse_dates=["signup_date"]
)

promotions = pd.read_csv(
    base_path + "promotions.csv",
    dtype={
        "promo_id": "string",
        "promo_name": "string",
        "promo_type": "string",
        "discount_value": "float64",
        "applicable_category": "string",
        "promo_channel": "string",
        "stackable_flag": "Int64",
        "min_order_value": "float64",
    },
    parse_dates=["start_date", "end_date"]
)

geography = pd.read_csv(
    base_path + "geography.csv",
    dtype={
        "zip": "Int64",
        "city": "string",
        "region": "string",
        "district": "string",
    }
)

orders = pd.read_csv(
    base_path + "orders.csv",
    dtype={
        "order_id": "Int64",
        "customer_id": "Int64",
        "zip": "Int64",
        "order_status": "string",
        "payment_method": "string",
        "device_type": "string",
        "order_source": "string",
    },
    parse_dates=["order_date"]
)

order_items = pd.read_csv(
    base_path + "order_items.csv",
    dtype={
        "order_id": "Int64",
        "product_id": "Int64",
        "quantity": "Int64",
        "unit_price": "float64",
        "discount_amount": "float64",
        "promo_id": "string",
        "promo_id_2": "string",
    }
)

payments = pd.read_csv(
    base_path + "payments.csv",
    dtype={
        "order_id": "Int64",
        "payment_method": "string",
        "payment_value": "float64",
        "installments": "Int64",
    }
)

shipments = pd.read_csv(
    base_path + "shipments.csv",
    dtype={
        "order_id": "Int64",
        "shipping_fee": "float64",
    },
    parse_dates=["ship_date", "delivery_date"]
)

returns = pd.read_csv(
    base_path + "returns.csv",
    dtype={
        "return_id": "string",
        "order_id": "Int64",
        "product_id": "Int64",
        "return_reason": "string",
        "return_quantity": "Int64",
        "refund_amount": "float64",
    },
    parse_dates=["return_date"]
)

reviews = pd.read_csv(
    base_path + "reviews.csv",
    dtype={
        "review_id": "string",
        "order_id": "Int64",
        "product_id": "Int64",
        "customer_id": "Int64",
        "rating": "Int64",
        "review_title": "string",
    },
    parse_dates=["review_date"]
)

sales = pd.read_csv(
    base_path + "sales.csv",
    dtype={
        "Revenue": "float64",
        "COGS": "float64",
    },
    parse_dates=["Date"]
)

inventory = pd.read_csv(
    base_path + "inventory.csv",
    dtype={
        "product_id": "Int64",
        "stock_on_hand": "Int64",
        "units_received": "Int64",
        "units_sold": "Int64",
        "stockout_days": "Int64",
        "days_of_supply": "float64",
        "fill_rate": "float64",
        "stockout_flag": "Int64",
        "overstock_flag": "Int64",
        "reorder_flag": "Int64",
        "sell_through_rate": "float64",
        "product_name": "string",
        "category": "string",
        "segment": "string",
        "year": "Int64",
        "month": "Int64",
    },
    parse_dates=["snapshot_date"]
)

web_traffic = pd.read_csv(
    base_path + "web_traffic.csv",
    dtype={
        "sessions": "Int64",
        "unique_visitors": "Int64",
        "page_views": "Int64",
        "bounce_rate": "float64",
        "avg_session_duration_sec": "float64",
        "conversion_rate": "float64",
        "traffic_source": "string",
    },
    parse_dates=["date"]
)


In [ ]:
# Q1. Trong số các khách hàng có nhiều hơn một đơn hàng, trung vị số ngày giữa hai lần
# mua liên tiếp (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ orders.csv)

df = orders.sort_values(['customer_id', 'order_date'])

df['prev_date'] = df.groupby('customer_id')['order_date'].shift(1)
df['gap_days'] = (df['order_date'] - df['prev_date']).dt.days

df = df.dropna(subset=['gap_days'])

valid_customers = df['customer_id'].unique()
df = df[df['customer_id'].isin(valid_customers)]

df['gap_days'].median()

# C) 144

In [ ]:
# Q2. Phân khúc sản phẩm (segment) nào trong products.csv có tỷ suất lợi nhuận gộp
# trung bình cao nhất, với công thức (price − cogs)/price?

sqldf('''
    SELECT segment, AVG(1.0 * (price - cogs) / price) AS avg_margin
    FROM products
    GROUP BY segment
    ORDER BY avg_margin DESC
''')

# D) Standard

In [ ]:
# Q3. Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục Streetwear (join
# returns với products theo product_id), lý do trả hàng nào xuất hiện nhiều nhất?

sqldf('''
    SELECT r.return_reason, COUNT(*) c
    FROM returns r
    JOIN products p ON p.product_id = r.product_id
    WHERE p.category = 'Streetwear'
    GROUP BY r.return_reason
    ORDER BY c DESC
''')

# B) wrong_size

In [ ]:
# Q4. Trong web_traffic.csv, nguồn truy cập (traffic_source) nào có tỷ lệ thoát trung
# bình (bounce_rate) thấp nhất trên tất cả các ngày xuất hiện nguồn đó trong cột traffic_source?

sqldf('''
    SELECT w.traffic_source
    FROM web_traffic w
    GROUP BY w.traffic_source
    ORDER BY AVG(w.bounce_rate) ASC
    LIMIT 1
''')

# C) email_campaign

In [ ]:
# Q5. Tỷ lệ phần trăm các dòng trong order_items.csv có áp dụng khuyến mãi (tức là promo_id
# không null) xấp xỉ là bao nhiêu?

sqldf('''
    SELECT 1.0 * COUNT(*) / (
        SELECT 1.0 * COUNT(*)
        FROM order_items o
    )
    FROM order_items o
    WHERE o.promo_id IS NOT NULL
''')

# C) 39

In [ ]:
# Q6. Trong customers.csv, xét các khách hàng có age_group khác null, nhóm tuổi nào có số
# đơn hàng trung bình trên mỗi khách hàng cao nhất? (tổng số đơn / số khách hàng trong
# nhóm)

sqldf('''
    SELECT c.age_group
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    WHERE age_group IS NOT NULL
    GROUP BY age_group
    ORDER BY COUNT(o.order_id) * 1.0 / COUNT(DISTINCT c.customer_id) DESC
    LIMIT 1
''')

# A) 55+

In [ ]:
# Q7. Vùng (region) nào trong geography.csv tạo ra tổng doanh thu cao nhất trong
# sales_train.csv?

sqldf('''
    SELECT g.region, SUM(s.Revenue) sum_revenue
    FROM geography g
    JOIN orders o ON o.zip = g.zip
    JOIN sales s ON s.Date = o.order_date
    WHERE s.Date BETWEEN '2012-07-04' AND '2022-12-31'
    GROUP BY g.region
    ORDER BY SUM(s.Revenue) DESC
    LIMIT 2
''')

# C) East

In [ ]:
# Q8. Trong các đơn hàng có order_status = ’cancelled’ trong orders.csv, phương thức
# thanh toán nào được sử dụng nhiều nhất?

sqldf('''
    SELECT p.payment_method
    FROM orders o
    JOIN payments p ON p.order_id = o.order_id
    WHERE o.order_status = 'cancelled'
    GROUP BY p.payment_method
    ORDER BY COUNT(*) DESC
    LIMIT 1
''')

# A) credit_card

In [ ]:
# Q9. Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có tỷ lệ trả hàng cao
# nhất, được định nghĩa là số bản ghi trong returns chia cho số dòng trong order_items (join
# với products theo product_id)?

sqldf('''
    SELECT 
        p.size,
        1.0 * COUNT(r.return_id) / COUNT(*) return_rate
    FROM order_items o
    JOIN products p ON o.product_id = p.product_id
    LEFT JOIN returns r ON o.product_id = r.product_id
    WHERE p.size IN ('S','M','L','XL')
    GROUP BY p.size
    ORDER BY return_rate DESC
''')

# D) XL

In [ ]:
# Q10. Trong payments.csv, kế hoạch trả góp nào có giá trị thanh toán trung bình trên
# mỗi đơn hàng cao nhất?

sqldf('''
    SELECT installments, AVG(payment_value) avg_payment
    FROM payments
    GROUP BY installments
    ORDER BY avg_payment DESC
    LIMIT 1
''')

# C) 6 kỳ